# Bài 3 — Xử lý lô và lưu trữ với PySpark
`bai3_batch.ipynb`

---

Xây dựng **lớp lô (batch layer)**: xử lý 30 ngày dữ liệu lịch sử, tạo các chỉ số tổng hợp phục vụ lớp phục vụ ở Bài 4.

**Mục tiêu:**
- Xử lý dữ liệu lớn bằng PySpark batch
- Aggregation theo nhiều chiều, dùng Window function
- Ghi Parquet có partition

## Phần A — Khởi tạo và khám phá dữ liệu

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

DATA_RAW    = os.environ.get("DATA_RAW_PATH",    "/data/raw")
DATA_OUTPUT = os.environ.get("DATA_OUTPUT_PATH", "/data/output")

spark = SparkSession.builder.appName("Bai3_Batch").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [ ]:
# TODO: đọc /data/raw/transactions_30days.parquet
# TODO: in schema, đếm bản ghi, kiểm tra null, describe amount
# TODO: đếm giao dịch theo category
# TODO: tính tỷ lệ is_fraud == True
df = ...

## Phần B — Chỉ số tổng hợp theo ngày

Tính theo `(date, category)` — chỉ giao dịch hợp lệ (`is_fraud=False`):

| Cột | Mô tả |
|---|---|
| `total_revenue` | Tổng amount hợp lệ |
| `txn_count` | Số giao dịch hợp lệ |
| `avg_amount` | Trung bình amount hợp lệ |
| `fraud_count` | Số giao dịch gian lận |
| `fraud_rate` | Tỷ lệ gian lận (làm tròn 4 chữ số) |

In [ ]:
def compute_daily_stats(df):
    # TODO: withColumn date từ timestamp
    # TODO: groupBy date, category
    # TODO: agg các chỉ số (dùng F.when để tính riêng hợp lệ / gian lận)
    # TODO: tính fraud_rate = fraud_count / total_count, làm tròn 4 chữ số
    pass


daily_stats = compute_daily_stats(df)
print(f"Rows: {daily_stats.count():,}")
daily_stats.show(5, truncate=False)

assert "fraud_rate"    in daily_stats.columns
assert "total_revenue" in daily_stats.columns
assert daily_stats.filter(F.col("total_revenue") < 0).count() == 0
print("✓ compute_daily_stats() hợp lệ")

## Phần C — Hồ sơ hành vi người dùng

Tính theo `user_id`:

| Cột | Mô tả |
|---|---|
| `total_spent` | Tổng chi tiêu hợp lệ |
| `txn_count` | Số giao dịch hợp lệ |
| `avg_amount` | Chi tiêu trung bình |
| `favourite_category` | Danh mục có tổng chi tiêu cao nhất |
| `days_active` | Số ngày khác nhau có giao dịch |

> `favourite_category` cần dùng Window function: tính amount theo `(user_id, category)`, lấy category rank=1.

In [ ]:
def compute_user_stats(df):
    valid = df.filter(F.col("is_fraud") == False)

    # TODO: tính base_stats (total_spent, txn_count, avg_amount, days_active) theo user_id
    base_stats = ...

    # TODO: tính favourite_category
    # Bước 1: tổng amount theo (user_id, category)
    # Bước 2: Window partitionBy user_id orderBy desc(cat_amount), lấy rank == 1
    # Bước 3: join với base_stats
    favourite = ...

    return base_stats.join(favourite, on="user_id", how="left")


user_stats = compute_user_stats(df)
print(f"Rows: {user_stats.count():,}")
user_stats.show(5, truncate=False)

assert "favourite_category" in user_stats.columns
assert user_stats.filter(F.col("favourite_category").isNull()).count() == 0
print("✓ compute_user_stats() hợp lệ")

## Phần D — Ghi kết quả

- `daily_stats` → `/data/output/batch/daily_stats/` — Parquet, partition theo `date`
- `user_stats` → `/data/output/batch/user_stats/` — Parquet, không partition
- Sau khi ghi: đọc lại và đếm để xác nhận

In [ ]:
DAILY_PATH = f"{DATA_OUTPUT}/batch/daily_stats"
USER_PATH  = f"{DATA_OUTPUT}/batch/user_stats"

# TODO: ghi daily_stats với partitionBy("date")
# TODO: ghi user_stats

# TODO: đọc lại và in số bản ghi
# TODO: in cây thư mục để xác nhận cấu trúc partition

## Kiểm tra đầu ra

In [ ]:
import os
print("=" * 45)
print("KIỂM TRA ĐẦU RA BÀI 3")
print("=" * 45)

checks = [
    ("daily_stats đủ 7 cột",
        all(c in daily_stats.columns for c in
            ["date","category","total_revenue","txn_count","avg_amount","fraud_count","fraud_rate"])),
    ("total_revenue không âm",
        daily_stats.filter(F.col("total_revenue") < 0).count() == 0),
    ("user_stats đủ 6 cột",
        all(c in user_stats.columns for c in
            ["user_id","total_spent","txn_count","avg_amount","favourite_category","days_active"])),
    ("favourite_category không null",
        user_stats.filter(F.col("favourite_category").isNull()).count() == 0),
    ("Partition date= tồn tại",
        any("date=" in d for _,dirs,_ in os.walk(f"{DATA_OUTPUT}/batch/daily_stats") for d in dirs)),
]
try:
    n = spark.read.parquet(f"{DATA_OUTPUT}/batch/daily_stats").count()
    checks.append(("Đọc lại daily_stats thành công", n > 0))
except: checks.append(("Đọc lại daily_stats thành công", False))

all_passed = True
for name, passed in checks:
    print(f"  [{'✓' if passed else '✗'}] {name}")
    if not passed: all_passed = False
print()
print("✓ Bài 3 hoàn thành!" if all_passed else "✗ Xem lại phần chưa qua.")